# Using `kugupu` to calculate molecular coupling networks

This notebook demonstrates how to calculate molecular coupling between fragments, inspect the results and save and load these results to file.  These results files will be the basis of all further analysis done using the `kugupu` package.

This will require version 0.20.0 of MDAnalysis, and kugupu to be installed.

In [15]:
import MDAnalysis as mda
import kugupu as kgp
import sys
import numpy as np
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "last_expr"
# np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(threshold=np.inf)  # only print up to 100 elements


Firstly we create an `MDAnalysis.Universe` object from our simulation files:

In [2]:
u = mda.Universe('datafiles/C6.data', 'datafiles/C6.dcd')

/Users/k2584788/.local/share/mamba/envs/forked_kugupu/lib/python3.10/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


This system has 46,500 atoms in 250 different fragments.

In [3]:
print(u.atoms.n_atoms, len(u.atoms.fragments))

46500 250


Our dynamics simulation has 5 frames of results.

In [4]:
print(u.trajectory.n_frames)

5


To perform the coupling calculations our `Universe` will require bond information (for determining fragments) and element information (for the tight binding calculations) stored inside the `.names` attribute.

Our Lammps Data file did not include element symbols, so we can add these to the Universe now...

In [5]:
def add_names(u):
    # Guesses atom names based upon masses
    def approx_equal(x, y):
        return abs(x - y) < 0.1
    
    # mapping of atom mass to element
    massdict = {}
    for m in set(u.atoms.masses):
        for elem, elem_mass in mda.guesser.tables.masses.items():
            if approx_equal(m, elem_mass):
                massdict[m] = elem
                break
        else:
            raise ValueError
            
    u.add_TopologyAttr('names')
    for m, e in massdict.items():
        u.atoms[u.atoms.masses == m].names = e

add_names(u)

## Running the coupling matrix calculation

The coupling matrix between fragments is calculated using the `kgp.coupling_matrix` function.

Here we are calculating the coupling matrix for fragments in the Universe `u` where
- coupling is calculated between fragments with a closest approach of less than 5.0 Angstrom (`nn_cutoff`)
- coupling is calculated between the LUMO upwards (`state='lumo'`)
- one state per fragment is considered (`degeneracy=1`)
- we will analyse up to frame 3 (`stop=3`)

This function will (for each frame)
- identify which fragments are close enough to possibly be electronically coupled
- run a tight binding calculation between all pairs identified
- calculate the molecular coupling based on this tight binding calculation

In [6]:
res = kgp.coupling_matrix(u, nn_cutoff=5.0, state='lumo', degeneracy=1, stop=1)

2025-06-19T20:42:47.873636+0100 INFO Processing 1 frames
2025-06-19T20:42:47.874519+0100 INFO Processing frame 1 of 1
2025-06-19T20:42:47.955939+0100 INFO Finding dimers within 5.0, passed 250 fragments
2025-06-19T20:42:48.307454+0100 INFO Found 3282 dimers
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_

no shift
for j bit the H_frag insert is [-10.48598052]
shift
for j bit the H_frag insert is [-10.33628492]
no shift
for j bit the H_frag insert is [-10.30981788]
no shift
for j bit the H_frag insert is [-10.40472391]
no shift
for j bit the H_frag insert is [-10.32165862]
no shift
for j bit the H_frag insert is [-10.38037008]
shift
for j bit the H_frag insert is [-10.42728372]
no shift


ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 368 and 365 (0.991499 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 235 and 215 (0.997515 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 370 and 369 (

for j bit the H_frag insert is [-10.42224309]
no shift
for j bit the H_frag insert is [-10.35147858]
shift
for j bit the H_frag insert is [-10.32410435]
shift
for j bit the H_frag insert is [-10.38891727]
shift
for j bit the H_frag insert is [-10.30041618]
shift
for j bit the H_frag insert is [-10.36044205]
no shift
for j bit the H_frag insert is [-10.410671]
no shift
for j bit the H_frag insert is [-10.33054855]
no shift
for j bit the H_frag insert is [-10.36497346]


ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 337 and 336 (

...

ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 304 and 303 (0.999701 A) is suspicious.
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using defau

no shift
shift
shift
no shift
no shift
shift
no shift
shift
no shift
no shift
no shift
no shift
no shift
no shift
no shift
no shift
no shift
shift
no shift
shift
no shift


ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/mini

no shift
no shift
no shift
no shift
shift
shift
shift
no shift
no shift
shift
no shift
no shift
no shift
no shift


ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
ERROR: Can't open parameter file: /Users/runner/miniforge3/conda-bld/yaehmop_1658385592334/work/tightbind/eht_parms.dat using default data in eht_parms.h....
!!! Warning !!! Distance between atoms 304 and 303 (

The `res` object is a namedtuple which contains all the data necessary to perform further analysis.
This object has various attributes which will not be briefly explained.

The `.frames` attribute records which frames from the trajectory were analysed.
This is useful to later cross reference data with the original MD trajectory data.

In [7]:
print(res.frames)

[0]


The `.degeneracy` attribute stores how many degenerate states were considered for each fragment.
This value will not change over time, so this array has shape `nfragments`.

In this example only a single state per fragment was considered. 

In [8]:
print(res.degeneracy)

[1 1 1 ... 1 1 1]


In [22]:
print(res.H_eff[0, 1, 71])

0.05520519011291175


The `.H_frag` attribute contains the molecular coupling values, stored inside a 3d numpy array.
The first dimension is along the number of frames (quasi time axis),
while the other two move along fragments in the system.

For example `res.H_frag[0, 1, 71]` gives the coupling (in eV) between the 2nd and 13th fragments in the first frame.

In [9]:
print(res.H_frag.shape)

print(res.H_frag[0, 1, 71])

(1, 250, 250)
0.03601076420417102


In [10]:
res.H_frag.shape

(1, 250, 250)

Producing these results is often a time consuming part of the analysis,
therefore it is wise to save them to a file so you can come back to them later!

This can be done using the `kugupu.save_results` function, which will save the results to a hdf5 (compressed) format.

In [11]:
# kgp.save_results('myresults.hdf5', res)

These results can then be retrieved again using the `kugupu.load_results` function:

In [12]:
kgp.load_results('./myresults.hdf5')

KeyError: "Unable to synchronously open object (object 'H_eff' doesn't exist)"